# TC9 ASP Long-Run Colab Notebook

This notebook stages the `DSE_Core` repo in Colab, runs the long TC9 ASP Phase 1 job, compares it against the MathOpt Phase 1 result, and downloads a zip of the resulting artifacts.

In [ ]:
!python -m pip install --quiet clingo==5.8.0 ortools pulp

## Repo Source

Use one of these options:

- Set `GIT_URL` if the repo is reachable from Colab.
- Otherwise upload a zip of the repo and point `REPO_ZIP` at it.

In [ ]:
from pathlib import Path

GIT_URL = ""  # e.g. https://github.com/<org>/<repo>.git
GIT_BRANCH = ""
REPO_ZIP = "/content/DesignSpaceExplorationforSecurity-main.zip"
WORK_ROOT = Path('/content/dse_workspace')
ASP_TIMEOUT_S = 0  # 0 = no timeout
MATHOPT_TIMEOUT_S = 120
CLINGO_THREADS = 8
CLINGO_PARALLEL_MODE = 'compete'
CLINGO_CONFIGURATION = None  # e.g. 'trendy'
CPSAT_THREADS = 1
SNAPSHOT_INTERVAL_S = 60

In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

WORK_ROOT.mkdir(parents=True, exist_ok=True)
repo_root = WORK_ROOT / 'repo'
if repo_root.exists():
    shutil.rmtree(repo_root)

if GIT_URL:
    clone_cmd = ['git', 'clone', GIT_URL, str(repo_root)]
    if GIT_BRANCH:
        clone_cmd = ['git', 'clone', '--branch', GIT_BRANCH, GIT_URL, str(repo_root)]
    subprocess.run(clone_cmd, check=True)
else:
    zip_path = Path(REPO_ZIP)
    if not zip_path.exists():
        raise FileNotFoundError(f'Repo zip not found: {zip_path}')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(repo_root)

candidates = list(repo_root.rglob('DSE_Core/tools/tc9_asp_colab_runner.py'))
if not candidates:
    raise FileNotFoundError('Could not find DSE_Core/tools/tc9_asp_colab_runner.py in the staged repo')
runner_path = candidates[0]
dse_core_root = runner_path.parents[2]
print('Runner:', runner_path)
print('DSE_Core:', dse_core_root)

In [ ]:
import json
import subprocess

artifact_dir = dse_core_root / 'tools' / 'colab_tc9_artifacts'
cmd = [
    'python',
    str(runner_path),
    '--output-dir', str(artifact_dir),
    '--strategy', 'max_security',
    '--asp-timeout', str(ASP_TIMEOUT_S),
    '--mathopt-timeout', str(MATHOPT_TIMEOUT_S),
    '--clingo-threads', str(CLINGO_THREADS),
    '--clingo-parallel-mode', CLINGO_PARALLEL_MODE,
    '--cpsat-threads', str(CPSAT_THREADS),
    '--snapshot-interval', str(SNAPSHOT_INTERVAL_S),
]
if CLINGO_CONFIGURATION:
    cmd.extend(['--clingo-configuration', CLINGO_CONFIGURATION])

completed = subprocess.run(cmd, check=True, cwd=str(dse_core_root), capture_output=True, text=True)
print(completed.stdout)
summary = json.loads(completed.stdout)
bundle_path = Path(summary['bundle_path'])
print('Bundle:', bundle_path)
print('Artifact dir:', artifact_dir)

In [ ]:
from pathlib import Path
import json

print((artifact_dir / 'README.md').read_text())
compare = json.loads((artifact_dir / 'compare_asp_vs_mathopt.json').read_text())
checkpoints = sorted(p.name for p in artifact_dir.glob('asp_checkpoint_*.json'))
print('Checkpoint files:', checkpoints[:10], '...' if len(checkpoints) > 10 else '')
compare

In [ ]:
from google.colab import files
files.download(str(bundle_path))